In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

import importlib

import ce_visualization_plotly.plugin as plotly_plugin

importlib.reload(plotly_plugin)
plotly_plugin.register_plotly_visualization_components()

# Live Plotly instance workspace

This notebook demonstrates Mode B: a live Python dashboard launched with `launch_instance_workspace(...)`.

Live mode keeps Python available. A selected instance can generate factual and alternative explanations on demand, add conjunction-aware cards, and build a workspace interactively. Unlike standalone HTML mode, this dashboard is not a single shareable precomputed file; it runs a local Python Dash application.

In [ ]:
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.dashboard import launch_instance_workspace
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Importing the package registers its Plotly styles with calibrated-explanations.

## Data

We use the same explicit 60/20/20 proper-training, calibration, and query split as in standalone mode.

In [ ]:
x, y = make_classification(
    n_samples=300,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=0,
)

x_train, x_query, y_train, y_query = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y,
)
x_proper, x_cal, y_proper, y_cal = train_test_split(
    x_train,
    y_train,
    test_size=0.25,
    random_state=0,
    stratify=y_train,
)

x_proper.shape, x_cal.shape, x_query.shape

## Fit and calibrate

Use `WrapCalibratedExplainer`; do not use `CalibratedExplainer` directly in this workflow.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=0)
explainer = WrapCalibratedExplainer(model)

explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

## Optional sanity plot

The live app starts with the current global instance explorer.

In [ ]:
global_result = explainer.plot(
    x_query,
    y_query,
    style="plotly.global.instance_explorer",
    task="classification",
    position_precision=2,
    show=True,
)

## Build the live dashboard app without blocking the notebook

The next cell constructs the Dash app and all callbacks but does not start the server. This is useful for checking that the app is configured in a notebook or test run. To actually open the dashboard, run the following launch cell after it.

In [ ]:
app = launch_instance_workspace(
    explainer,
    x_query,
    y_query,
    available_cards="auto",
    task="classification",
    max_rule_size_default=3,
    default_filter_top=20,
    host="127.0.0.1",
    port=8050,
    debug=False,
    open_browser=False,
    run_server=False,
    include_conjunctions=True,
)

app.ce_workspace_url

## Launch the live dashboard

Run this cell when you want to start the local Dash server. It blocks the current kernel while the server is running. Stop the cell or interrupt the kernel to stop the dashboard.

In [ ]:
app = launch_instance_workspace(
    explainer,
    x_query,
    y_query,
    available_cards="auto",
    task="classification",
    max_rule_size_default=3,
    default_filter_top=20,
    host="127.0.0.1",
    port=8050,
    debug=False,
    open_browser=True,
    include_conjunctions=True,
)

## What live mode can do

Live mode can ask Python to generate factual and alternative explanations for selected instances while the dashboard is running. The workspace cards use the currently implemented Plotly card styles: `plotly.local.uncertainty_quadrant`, `plotly.local.ensured`, and `plotly.local.alternative_feature_summary`.

Live mode is for interactive local analysis. Use standalone mode when you need a shareable precomputed HTML artifact.